In [331]:
import pandas as pd
import numpy as np
from scipy.stats import entropy

# Calculate Generalized Fish Metrics from CCFRP Data
## Prepare Data
### I. Import Raw Dataframe
Import and clean catch data from the California Collaborative Fisheries Research Program. 

In [332]:
df = pd.read_csv('CCFRP_Data/RAW_CCFRP_DATA.csv')

### II. Extract Relevant Metadata

In [333]:
# Extract area of each grid cell from the drift ID 
#e.g. The 'AI' in AIM0908200201 corresponds to the unique area code for Anacapa Island
df.loc[:, 'Area'] = df['Grid_Cell_ID'].str[0:2]

# Extract date of sampling from the drift ID
# e.g. '090820' in AIM0908200201 denotes the day that unique drift occured in a ddmmyy format  
df.loc[:, 'Date'] = df['Drift_ID'].str[3:9]

# Extract protection status of each grid cell from the drift ID 
# e.g. The 'M' in AIM0908200201 denotes MPA 
df.loc[:, 'Protection'] = 'REF'
df.loc[df['Drift_ID'].str[2] == 'M', 'Protection'] = 'MPA'

# Remove all grid cell ID's that end in "MM" or "RR", which indicates that the drift should be voided 
df = df[~df['Grid_Cell_ID'].str.endswith(('MM', 'RR'), na=False)].copy()

## Summarize Generalized Fish Metrics
### I. Calculate Catch Per Unit Effort

Calculate catch per unit effort (CPUE) for each unqiue fishing drift. 

In [334]:
# Sum total catch per drift
total_catch = df.groupby(['Drift_ID', 'Grid_Cell_ID', 'Date', 'Year', 'Protection', 'Area']).size().reset_index(name='Total_Catch')

# Identify total angler hours per drift (assuming constant within drift)
angler_hours = df.groupby('Drift_ID')['Total_Angler_Hrs'].first().reset_index()

# Calculate catch per unit effort (total catch / angler hours) for each drift
cpue_df = pd.merge(total_catch, angler_hours, on='Drift_ID')
cpue_df['drift_CPUE'] = cpue_df['Total_Catch'] / cpue_df['Total_Angler_Hrs']

In [335]:
output_path = "CCFRP_Data/Outputs/ccfrp_cpue_summary.csv"
cpue_df.to_csv(output_path, index=False)

### II. Calculate Biodiversity

In [336]:
# Define a function that computes species richness and Shannon diversity
def compute_diversity(group):
    species_counts = group['Species_Code'].value_counts()
    proportions = species_counts / species_counts.sum()
    
    return pd.Series({
        'drift_richness': species_counts.nunique(),
        'drift_shannon_index': entropy(proportions)
    })

# Apply the function to each drift 
diversity_df = df.groupby(['Drift_ID', 'Grid_Cell_ID', 'Date', 'Year', 'Protection', 'Area']).apply(compute_diversity).reset_index()

C:\Users\FELAB\AppData\Local\Temp\ipykernel_34484\1555288053.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  diversity_df = df.groupby(['Drift_ID', 'Grid_Cell_ID', 'Date', 'Year', 'Protection', 'Area']).apply(compute_diversity).reset_index()


In [337]:
output_path = "CCFRP_Data/Outputs/ccfrp_biodiversity_summary.csv"
diversity_df.to_csv(output_path, index=False)

### III. Calculate Biomass

In [338]:
# biomass_df

## Export Summary Table

Calculate the average CPUE and diversity for each instance a grid cell is sampled. In some cases, the same grid cell might be sampled multiple times a year because grid cells are sampled randomly. Therefore, each unique day that a grid cell is sampled must be regarded as an independent event. 

In [339]:
combined_df = pd.merge(cpue_df, diversity_df, on=['Drift_ID', 'Grid_Cell_ID', 'Date', 'Year', 'Protection', 'Area'])

In [340]:
master_summary = (
    combined_df.groupby(['Grid_Cell_ID', 'Date', 'Year', 'Protection', 'Area'])
    .agg(
        mean_cpue=('drift_CPUE', 'mean'),
        mean_richness=('drift_richness', 'mean'), 
        mean_shannon_index=('drift_shannon_index', 'mean')
    )
    .reset_index()
)

In [341]:
output_path = "CCFRP_Data/Outputs/ccfrp_master_summary.csv"
master_summary.to_csv(output_path, index=False)

master_summary

,Grid_Cell_ID,Date,Year,Protection,Area,mean_cpue,mean_richness,mean_shannon_index
0,AI01,091520,2020,MPA,AI,18.747181,2.750000,0.548642
1,AI01,091819,2019,MPA,AI,21.785935,1.800000,0.139709
2,AI01,091919,2019,MPA,AI,20.220319,3.000000,0.480670
3,AI01,092121,2021,MPA,AI,12.621540,3.666667,0.811692
4,AI01,100422,2022,MPA,AI,12.946308,2.000000,0.351485
...,...,...,...,...,...,...,...,...
3644,TM35,081820,2020,MPA,TM,6.221741,1.333333,0.675109
3645,TM35,090517,2017,MPA,TM,3.255316,1.333333,0.809425
3646,TM35,090721,2021,MPA,TM,13.228494,2.333333,0.809359
3647,TM35,091022,2022,MPA,TM,12.195547,2.333333,0.727676
